In [1]:
import pandas as pd
import json

# =========================================================
# 1. DOSYA YOLLARI
# =========================================================
PATH_RAW = '../data/raw/'

# =========================================================
# 2. VERİLERİ OKUMA
# =========================================================
users = pd.read_csv(f'{PATH_RAW}users_data.csv')
cards = pd.read_csv(f'{PATH_RAW}cards_data.csv')

In [2]:
chunks = []

for chunk in pd.read_csv(f"{PATH_RAW}transactions_data.csv", chunksize=200000):
    sample = chunk.sample(min(5000, len(chunk)), random_state=42)
    chunks.append(sample)
transactions = pd.concat(chunks, ignore_index=True)

In [3]:
# Büyük dosya olduğu için şimdilik ilk 100.000 satır
#transactions = pd.read_csv(f'{PATH_RAW}transactions_data.csv')

with open(f'{PATH_RAW}mcc_codes.json', 'r') as f:
    mcc_codes = json.load(f)

with open(f'{PATH_RAW}train_fraud_labels.json', 'r') as f:
    fraud_labels = json.load(f)

print("✅ Tüm veriler başarıyla yüklendi!")

✅ Tüm veriler başarıyla yüklendi!


In [7]:
# 3. İLK KONTROLLER
# =========================================================
print("\n--- Veri Boyutları ---")
print(f"Users shape: {users.shape}")
print(f"Cards shape: {cards.shape}")
print(f"Transactions shape: {transactions.shape}")

print("\n--- Transactions sütunları ---")
print(transactions.columns.tolist())

print("\n--- Users sütunları ---")
print(users.columns.tolist())

print("\n--- Cards sütunları ---")
print(cards.columns.tolist())



--- Veri Boyutları ---
Users shape: (2000, 14)
Cards shape: (6146, 13)
Transactions shape: (335000, 13)

--- Transactions sütunları ---
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_return']

--- Users sütunları ---
['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']

--- Cards sütunları ---
['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web']


In [4]:
# 4. TRANSACTIONS TEMİZLİĞİ
# =========================================================

# Amount: '$' işaretini kaldır, float yap
transactions['amount'] = (
    transactions['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Date: datetime yap
transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')

# Zip: string yap, .0 temizle
transactions['zip'] = (
    transactions['zip']
    .astype(str)
    .str.replace('.0', '', regex=False)
)

# Negatif tutarlar iade olabilir
transactions['is_return'] = transactions['amount'] < 0
transactions['amount'] = transactions['amount'].abs()

print("\n✅ Transactions temizliği tamamlandı!")



✅ Transactions temizliği tamamlandı!


In [5]:
# 5. ID TİPLERİNİ EŞİTLE
# =========================================================
users['id'] = users['id'].astype(str)
cards['id'] = cards['id'].astype(str)
cards['client_id'] = cards['client_id'].astype(str)

transactions['id'] = transactions['id'].astype(str)
transactions['client_id'] = transactions['client_id'].astype(str)
transactions['card_id'] = transactions['card_id'].astype(str)

print("\n--- ID tipleri ---")
print("users['id']:", users['id'].dtype)
print("cards['id']:", cards['id'].dtype)
print("cards['client_id']:", cards['client_id'].dtype)
print("transactions['id']:", transactions['id'].dtype)
print("transactions['client_id']:", transactions['client_id'].dtype)
print("transactions['card_id']:", transactions['card_id'].dtype)


--- ID tipleri ---
users['id']: object
cards['id']: object
cards['client_id']: object
transactions['id']: object
transactions['client_id']: object
transactions['card_id']: object


In [6]:
# 6. TRANSACTIONS + USERS MERGE
# =========================================================
df_combined = pd.merge(
    transactions,
    users,
    left_on='client_id',
    right_on='id',
    how='left',
    suffixes=('', '_user')
)

print("\n✅ Transactions + Users birleştirildi")
print("Shape:", df_combined.shape)


✅ Transactions + Users birleştirildi
Shape: (335000, 27)


In [7]:
# users tablosundan gelen id sütununu kaldır
if 'id_user' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_user'])

# Bazı durumlarda merge sonrası users.id direkt 'id_y' olabilir
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])


In [8]:
# 7. CARDS TABLOSUNU EKLE
# transactions.card_id  <-> cards.id
# =========================================================
df_combined = pd.merge(
    df_combined,
    cards,
    left_on='card_id',
    right_on='id',
    how='left',
    suffixes=('', '_card')
)

print("\n✅ Cards tablosu eklendi")
print("Shape:", df_combined.shape)

# cards'tan gelen gereksiz tekrar id sütununu kaldır
if 'id_card' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_card'])

# Eğer merge sonrası 'id_y' oluştuysa temizle
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])



✅ Cards tablosu eklendi
Shape: (335000, 39)


In [9]:
# 8. MCC CODES EKLEME
# =========================================================
# mcc_codes JSON ise dict olarak gelir:
# örnek: {"1711": "Heating, Plumbing...", ...}

mcc_df = pd.DataFrame(list(mcc_codes.items()), columns=['mcc', 'mcc_description'])
mcc_df['mcc'] = mcc_df['mcc'].astype(str)

# transactions içindeki mcc de string olsun
df_combined['mcc'] = df_combined['mcc'].astype(str)

df_combined = pd.merge(
    df_combined,
    mcc_df,
    on='mcc',
    how='left'
)

print("\n✅ MCC açıklamaları eklendi")
print("Shape:", df_combined.shape)



✅ MCC açıklamaları eklendi
Shape: (335000, 39)


In [10]:
# 9. FRAUD LABELS DÜZELTME VE EKLEME
# =========================================================
# JSON yapısı:
# {
#   "target": {
#       "7476734": "No",
#       "7481767": "Yes",
#       ...
#   }
# }

fraud_labels = fraud_labels['target']

fraud_df = pd.DataFrame(list(fraud_labels.items()), columns=['id', 'is_fraud'])
fraud_df['id'] = fraud_df['id'].astype(str)

# Yes/No -> 1/0
fraud_df['is_fraud'] = fraud_df['is_fraud'].map({'Yes': 1, 'No': 0})

# Fraud label transaction id üzerinden eklenmeli
# Çünkü fraud JSON key = transactions.id
df_combined = pd.merge(
    df_combined,
    fraud_df,
    on='id',
    how='left'
)

# Eksik olanları 0 kabul et
df_combined['is_fraud'] = df_combined['is_fraud'].fillna(0).astype(int)

print("\n✅ Fraud label eklendi")
print("Shape:", df_combined.shape)


✅ Fraud label eklendi
Shape: (335000, 40)


In [11]:
# 10. TARİH FEATURE'LARI
# =========================================================
df_combined['year'] = df_combined['date'].dt.year
df_combined['month'] = df_combined['date'].dt.month
df_combined['day'] = df_combined['date'].dt.day
df_combined['hour'] = df_combined['date'].dt.hour
df_combined['day_of_week'] = df_combined['date'].dt.day_name()
df_combined['is_weekend'] = df_combined['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

print("\n✅ Tarih feature'ları oluşturuldu")


✅ Tarih feature'ları oluşturuldu


In [12]:
# 11. KONTROLLER
# =========================================================
print("\n--- Son tablo bilgisi ---")
print(df_combined.shape)

print("\n--- İlk 5 satır ---")
print(df_combined.head())

print("\n--- Fraud dağılımı ---")
print(df_combined['is_fraud'].value_counts(dropna=False))

print("\n--- Fraud oranı ---")
print(df_combined['is_fraud'].mean())

print("\n--- Eksik değer kontrolü (ilk 20 sütun) ---")
print(df_combined.isnull().sum().sort_values(ascending=False).head(20))



--- Son tablo bilgisi ---
(335000, 46)

--- İlk 5 satır ---
        id                date client_id card_id  amount            use_chip  \
0  7618350 2010-02-06 14:13:00       639    5016    1.29   Swipe Transaction   
1  7561682 2010-01-23 08:36:00       914    5835   94.00   Swipe Transaction   
2  7664295 2010-02-17 23:45:00       564    5887   45.15  Online Transaction   
3  7553437 2010-01-21 01:13:00      1682    3407   43.09   Swipe Transaction   
4  7511314 2010-01-10 09:03:00      1662    4541   26.35   Swipe Transaction   

   merchant_id merchant_city merchant_state    zip  ... year_pin_last_changed  \
0        14528      Sandusky             OH  44870  ...                  2010   
1        61195       Wichita             KS  67212  ...                  2007   
2        42332        ONLINE            NaN    nan  ...                  2011   
3        34702   Devils Lake             ND  58301  ...                  2011   
4        55188    Belleville             IL  62221  .

In [29]:
df_combined['is_fraud'].unique()

array([0, 1])

In [13]:
df_combined['is_fraud'].sum()

334

In [31]:
df_combined

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,year_pin_last_changed,card_on_dark_web,mcc_description,is_fraud,year,month,day,hour,day_of_week,is_weekend
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,2008,No,Miscellaneous Food Stores,0,2010,1,1,0,Friday,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,2015,No,Department Stores,0,2010,1,1,0,Friday,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,2008,No,Money Transfer,0,2010,1,1,0,Friday,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,2006,No,Money Transfer,0,2010,1,1,0,Friday,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,2014,No,Drinking Places (Alcoholic Beverages),0,2010,1,1,0,Friday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13305910,23761868,2019-10-31 23:56:00,1718,2379,1.11,Chip Transaction,86438,West Covina,CA,91792,...,2019,No,Miscellaneous Food Stores,0,2019,10,31,23,Thursday,0
13305911,23761869,2019-10-31 23:56:00,1766,2066,12.80,Online Transaction,39261,ONLINE,NaN,nan,...,2012,No,"Digital Goods - Media, Books, Apps",0,2019,10,31,23,Thursday,0
13305912,23761870,2019-10-31 23:57:00,199,1031,40.44,Swipe Transaction,2925,Allen,TX,75002,...,2007,No,"Utilities - Electric, Gas, Water, Sanitary",0,2019,10,31,23,Thursday,0
13305913,23761873,2019-10-31 23:58:00,1986,5443,4.00,Chip Transaction,46284,Daly City,CA,94014,...,2010,No,"Grocery Stores, Supermarkets",0,2019,10,31,23,Thursday,0


In [18]:
df_combined.to_csv("../data/processed/combined_chunked.csv")

In [ ]:
#df_full_combined=pd.read_csv("../data/processed/final_table.csv")

In [13]:
import numpy as np

In [20]:
df_combined['amount'] = (
    df_combined['amount']
    .astype(str)
    .str.replace(r"[^0-9\.\-]", "", regex=True)
    .astype(float)
)
df_combined['credit_limit'] = (
    df_combined['credit_limit']
    .astype(str)
    .str.replace(r"[^0-9\.]", "", regex=True)
    .astype(float)
)
df_combined['errors'] = (
    df_combined['errors']
    .astype(str)
    .str.strip()
    .str.lower()
)

df_combined['errors'] = df_combined['errors'].replace('nan', 'no_error')
df_combined['merchant_city'] = (
    df_combined['merchant_city']
    .astype(str)
    .str.strip()
    .str.lower()
)
df_combined['merchant_state'] = (
    df_combined['merchant_state']
    .astype(str)
    .str.strip()
    .str.upper()
)
df_combined['merchant_id'] = df_combined['merchant_id'].astype(str).str.strip()
df_combined['card_type'] = (
    df_combined['card_type']
    .astype(str)
    .str.strip()
    .str.lower()
)
df_combined['client_id'] = df_combined['client_id'].astype(str).str.strip()

/tmp/ipykernel_5569/2646388972.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/2646388972.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .apply(lambda x: x.sort_values('date')
/tmp/ipykernel_5569/2646388972.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/2646388972.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .apply(lambda x: x.sort_values('date')
/tmp/ipykernel_5569/2646388972.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/2646388972.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' ins

In [21]:


# datetime
df_combined['date'] = pd.to_datetime(df_combined['date'])

# sıralama önemli!
df_combined = df_combined.sort_values(['client_id', 'date'])

############################################
# 1. ZAMAN BAZLI FEATURELAR
############################################

# önceki işlemden geçen süre (dakika)
df_combined['time_diff'] = df_combined.groupby('client_id')['date'].diff().dt.total_seconds() / 60

# aynı gün işlem sayısı
df_combined['txn_per_day'] = df_combined.groupby(
    ['client_id', df_combined['date'].dt.date]
)['id'].transform('count')



############################################
# 2. HARCAMA DAVRANIŞI
############################################

# müşteri ortalama harcaması
df_combined['avg_amount_user'] = df_combined.groupby('client_id')['amount'].transform('mean')
df_combined["log_amount"] = np.log1p(df_combined["amount"])
# harcama sapması
df_combined['amount_deviation'] = df_combined['amount'] - df_combined['avg_amount_user']

# z-score
df_combined['amount_zscore'] = (
    (df_combined['amount'] - df_combined.groupby('client_id')['amount'].transform('mean')) /
    df_combined.groupby('client_id')['amount'].transform('std')
)
df_combined["high_amount"] = (df_combined["amount"] > df_combined["amount"].quantile(0.95)).astype(int)
############################################
# 3. LOKASYON SAPMASI
############################################

# müşterinin en sık bulunduğu şehir
most_common_city = df_combined.groupby('client_id')['merchant_city'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
)

df_combined['home_city'] = df_combined['client_id'].map(most_common_city)

# şehir değişimi
df_combined['is_foreign_city'] = (df_combined['merchant_city'] != df_combined['home_city']).astype(int)

############################################
# 4. HIZLI İŞLEM (BURST ACTIVITY)
############################################

# son 1 saat içindeki işlem sayısı
'''df_combined['txn_last_1h'] = df_combined.groupby('client_id')['date'].transform(
    lambda x: x.rolling('1H').count()
)'''
df_combined['txn_last_1h'] = (
    df_combined
    .groupby('client_id', group_keys=False)
    .apply(lambda x: x.sort_values('date')
           .rolling('1H', on='date')['amount']
           .count())
)
df_combined["time_diff"] = df_combined.groupby("client_id")["date"].diff().dt.total_seconds()

df_combined["fast_tx"] = (df_combined["time_diff"] < 60).astype(int)        # 1 dk
df_combined["very_fast_tx"] = (df_combined["time_diff"] < 10).astype(int)   # 10 sn
############################################
# 5. MERCHANT DAVRANIŞI
############################################

# farklı merchant sayısı (son 10 işlem)
df_combined['unique_merchants_last10'] = df_combined.groupby('client_id')['merchant_id'].transform(
    lambda x: x.rolling(10).apply(lambda y: len(set(y)))
)

############################################
# 6. KART RİSKİ
############################################

# dark web flag
df_combined['card_darkweb_flag'] = df_combined['card_on_dark_web']

# kredi limiti kullanım oranı
df_combined['credit_utilization'] = df_combined['amount'] / df_combined['credit_limit']

############################################
# 7. RETURN & ERROR
############################################

df_combined['has_error'] = df_combined['errors'].notna().astype(int)

############################################
# ---------------------------
# 1. TEMİZLİK
# ---------------------------
#df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# tarih parse (emin olmak için)

# gece / yoğun saat
df_combined["is_night"] = ((df_combined["hour"] < 6) | (df_combined["hour"] > 22)).astype(int)
df_combined["is_peak_hour"] = df_combined["hour"].isin([12,13,18,19,20]).astype(int)

# hafta sonu zaten var ama garanti:
df_combined["is_weekend"] = df_combined["date"].dt.weekday >= 5

df_combined.head()

/tmp/ipykernel_5569/708893512.py:63: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/708893512.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .apply(lambda x: x.sort_values('date')
/tmp/ipykernel_5569/708893512.py:63: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/708893512.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .apply(lambda x: x.sort_values('date')
/tmp/ipykernel_5569/708893512.py:63: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling('1H', on='date')['amount']
/tmp/ipykernel_5569/708893512.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' ins

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,is_foreign_city,txn_last_1h,fast_tx,very_fast_tx,unique_merchants_last10,card_darkweb_flag,credit_utilization,has_error,is_night,is_peak_hour
1130,7479264,2010-01-01 22:13:00,0,4639,65.86,Swipe Transaction,32164,scarborough,ME,4074,...,0,1.0,0,0,NaN,No,0.003742,1,0,0
888,7582582,2010-01-28 11:34:00,0,4639,83.26,Swipe Transaction,24823,scarborough,ME,4074,...,0,1.0,0,0,NaN,No,0.004731,1,0,0
4206,7608231,2010-02-03 22:05:00,0,4639,54.56,Swipe Transaction,32164,scarborough,ME,4074,...,0,1.0,0,0,NaN,No,0.003100,1,0,0
1289,7630461,2010-02-09 12:53:00,0,4639,88.00,Swipe Transaction,43293,scarborough,ME,4074,...,0,1.0,0,0,NaN,No,0.005000,1,0,1
9693,7726494,2010-03-05 14:02:00,0,4639,345.16,Online Transaction,86286,online,NAN,nan,...,1,1.0,0,0,NaN,No,0.019611,1,0,0


In [26]:
category_risk_map = df_combined.groupby('mcc_description')['is_fraud'].mean().to_dict()
df_combined['category_risk_score'] = df_combined['mcc_description'].map(category_risk_map)
user_avg_amount = df_combined.groupby('client_id')['amount'].transform('mean')

# 2. Harcama Sapması (Amount to Mean Ratio): Harcamanın kullanıcı ortalamasına oranı
df_combined['amount_ratio'] = df_combined['amount'] / user_avg_amount

In [19]:

'''

# ---------------------------
# 3. AMOUNT FEATURES
# ---------------------------
if "amount" in df_combined.columns:
    df_combined["log_amount"] = np.log1p(df_combined["amount"])
    df_combined["amount_zscore"] = (df_combined["amount"] - df_combined["amount"].mean()) / df_combined["amount"].std()

    # yüksek işlem flag
    df_combined["high_amount"] = (df_combined["amount"] > df_combined["amount"].quantile(0.95)).astype(int)

# ---------------------------
# 4. USER BEHAVIOR (ÇOK KRİTİK)
# ---------------------------
if "client_id" in df_combined.columns:

    df_combined = df_combined.sort_values(["client_id", "date"])

    # kullanıcı istatistikleri
    df_combined["user_tx_count"] = df_combined.groupby("client_id")["id"].transform("count")

    if "amount" in df_combined.columns:
        df_combined["user_mean_amount"] = df_combined.groupby("client_id")["amount"].transform("mean")
        df_combined["user_std_amount"] = df_combined.groupby("client_id")["amount"].transform("std")

        # anomali
        df_combined["amount_deviation"] = df_combined["amount"] - df_combined["user_mean_amount"]

# ---------------------------
# 5. TRANSACTION VELOCITY 🔥
# ---------------------------
if "client_id" in df_combined.columns:
    df_combined["time_diff"] = df_combined.groupby("client_id")["date"].diff().dt.total_seconds()

    df_combined["fast_tx"] = (df_combined["time_diff"] < 60).astype(int)        # 1 dk
    df_combined["very_fast_tx"] = (df_combined["time_diff"] < 10).astype(int)   # 10 sn

# ---------------------------
# 6. ROLLING FEATURES
# ---------------------------
if "client_id" in df_combined.columns and "amount" in df_combined.columns:
    df_combined["rolling_mean_3"] = df_combined.groupby("client_id")["amount"].transform(lambda x: x.rolling(3).mean())
    df_combined["rolling_std_3"] = df_combined.groupby("client_id")["amount"].transform(lambda x: x.rolling(3).std())
'''

In [27]:
print(df_combined.columns)

Index(['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'is_return', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'latitude', 'longitude',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'client_id_card', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'mcc_description', 'is_fraud', 'year', 'month',
       'day', 'hour', 'day_of_week', 'is_weekend', 'time_diff', 'txn_per_day',
       'avg_amount_user', 'log_amount', 'amount_deviation', 'amount_zscore',
       'high_amount', 'home_city', 'is_foreign_city', 'txn_last_1h', 'fast_tx',
       'very_fast_tx', 'unique_merchants_last10', 'card_darkweb_flag',
       'credit_utilization', 'has_error', 'is_nigh

In [28]:
df_combined.to_parquet("../data/processed/df_combined_eda_sample.parquet", index=False)

In [30]:
df_combined.to_csv("../data/processed/df_combined_eda_sample.csv", index=False)

In [29]:
print(df_combined["is_fraud"].value_counts())

is_fraud
0    334666
1       334
Name: count, dtype: int64
